START -> Reasearch -> Write -> human review -> finalize -> END

In [ ]:
from typing import TypedDict, Annotated
import operator

class ReportState(TypedDict):
    topic : str
    draft : str
    report : str
    feedback : Annotated[list[str],operator.add]
    final_report : str
    revision_cnt : int

In [ ]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm_research = ChatGroq(model = "openai/gpt-oss-20b")
llm_write = ChatGroq(model = "openai/gpt-oss-20b")

In [ ]:
def research_agent(state : ReportState):
    print("research_agententered")
    message = f"You are a expert in researching. Please give the important points for given topic {state['topic']}. These points should be grouped by suitable headings."
    draft = llm_research.invoke(message).content
    
    return {
        "draft" : draft
    }


In [ ]:
def write_agent(state : ReportState):
    print("write_agent entered")
    message = f"You are a expert in writing report. Please write the report for given topic {state['topic']} using draft {state['draft']}. Consider feedback {state['feedback']} if any."
    report = llm_write.invoke(message).content
    
    return {
        "report" : report
    }



In [ ]:
from langgraph.types import interrupt,Command
from typing import Literal
def human_review(state : ReportState) -> Command[Literal["finalize","write_agent"]]:
    print("human review entered")
    decision = interrupt({"question":"Please approve, edit or reject with feedback","payload":state["report"]})

    if decision["action"] == "approve":
        return Command(goto = "finalize")
    elif decision["action"] == "edit":
        return Command(goto = "finalize",update = {"report":decision["editedReport"]})
    else :
        resvision_cnt = state.get("revision_cnt",0) +1
        if  resvision_cnt <= 3 :
            print("*******",resvision_cnt)
            return Command(goto = "write_agent", update = {"feedback":[decision["feedback"]],"revision_cnt":resvision_cnt})
        return Command(goto = "finalize")
    
    return ValueError("Invalid option")

def finalize(state: ReportState):
    print("finalize entered")
    return {
        "final_report" : state["report"]
    }


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

build = StateGraph(state_schema=ReportState)

# nodes
build.add_node("research_agent",research_agent)
build.add_node("write_agent",write_agent)
build.add_node("human_review",human_review)
build.add_node("finalize",finalize)

# add edges 
build.add_edge(START, "research_agent")
build.add_edge("research_agent","write_agent")
build.add_edge("write_agent","human_review")
build.add_edge("finalize", END)

checkPointer = InMemorySaver()
workflow = build.compile(checkpointer=checkPointer)

config = {"configurable":{"thread_id":"user123"}}

In [ ]:
intitial_state = {"topic":"AI in health care"}

final_state = workflow.invoke(intitial_state,config=config)

In [ ]:
final_state

In [ ]:
# workflow.invoke(Command(resume={"action":"approve"}),config=config)

In [ ]:
# workflow.invoke(Command(resume={"action":"edit","editedReport":"edited......"}),config=config)

In [ ]:
workflow.invoke(Command(resume={"action":"reject","feedback":"feedback1"}),config=config)

In [ ]:
workflow.invoke(Command(resume={"action":"reject","feedback":"feedback2"}),config=config)

In [ ]:
workflow.invoke(Command(resume={"action":"reject","feedback":"feedback3"}),config=config)

In [ ]:
workflow.invoke(Command(resume={"action":"reject","feedback":"feedback4"}),config=config)